### Phase 1: Data Collection (Nifty 100 - From 1-1-2020 till date)

In [1]:
# ==========================================
# 📊 Phase 1: Data Collection (NIFTY 100)
# Source: NSE India (via nselib)
# ==========================================

from nselib import capital_market
import pandas as pd
from datetime import date, datetime
import time
import requests
import feedparser
from bs4 import BeautifulSoup
from urllib.parse import quote_plus

%load_ext nb_black

<IPython.core.display.Javascript object>

In [2]:
# -----------------------------
# Step 0: Project Path Setup (CRITICAL FIX)
# -----------------------------
from pathlib import Path

# Notebook lives inside /notebooks
# So project root is one level up
PROJECT_ROOT = Path().resolve().parent
DATA_DIR = PROJECT_ROOT / "data"

RAW_DIR = DATA_DIR / "raw"
NEWS_DIR = DATA_DIR / "news_raw"
CONCALL_DIR = DATA_DIR / "concall_raw"

RAW_DIR.mkdir(parents=True, exist_ok=True)
NEWS_DIR.mkdir(parents=True, exist_ok=True)
CONCALL_DIR.mkdir(parents=True, exist_ok=True)


<IPython.core.display.Javascript object>

In [3]:
import shutil

# Clear RAW directory to avoid leftover old files
if RAW_DIR.exists():
    shutil.rmtree(RAW_DIR)

RAW_DIR.mkdir(parents=True, exist_ok=True)

<IPython.core.display.Javascript object>

In [4]:
print(f"📁 Project root: {PROJECT_ROOT}")
print("📁 Data directories ready.")

📁 Project root: C:\Users\jayan\Study Materials\ml_alpha_portfolio\ml-alpha-portfolio-model
📁 Data directories ready.


<IPython.core.display.Javascript object>

In [5]:
from transformers import pipeline

sentiment_model = pipeline(
    "sentiment-analysis",
    model="ProsusAI/finbert",
    framework="pt",
)

print("✅ FinBERT loaded.")

Device set to use cpu


✅ FinBERT loaded.


<IPython.core.display.Javascript object>

In [6]:
# --- STEP 1: Fetch NIFTY 100 tickers ---
url = "https://archives.nseindia.com/content/indices/ind_nifty100list.csv"
df_list = pd.read_csv(url)
tickers = df_list["Symbol"].tolist()

print(f"✅ Total tickers fetched: {len(tickers)}")

✅ Total tickers fetched: 100


<IPython.core.display.Javascript object>

In [7]:
# --- STEP 2: Create news data folder ---
def fetch_news_sentiment(symbol, max_items=50):

    query = f"{symbol} stock site:reuters.com OR site:economictimes.com OR site:moneycontrol.com"
    query_encoded = quote_plus(query)

    url = f"https://news.google.com/rss/search?q={query_encoded}&hl=en-IN&gl=IN&ceid=IN:en"
    feed = feedparser.parse(url)

    records = []

    for entry in feed.entries[:max_items]:

        if not hasattr(entry, "published_parsed"):
            continue

        text = entry.title
        date_pub = datetime(*entry.published_parsed[:6])

        try:
            sent = sentiment_model(text)[0]
            score = sent["score"] if sent["label"] == "positive" else -sent["score"]
        except:
            continue

        records.append(
            {
                "ticker": symbol,
                "date": date_pub.date(),
                "headline": text,
                "sentiment_score": score,
            }
        )

    return pd.DataFrame(records)

<IPython.core.display.Javascript object>

In [8]:
# --- STEP 3: Fetch function ---
def fetch_all_nselib_data(symbols, start, end):

    failed = []

    for symbol in symbols:

        try:
            print(f"📦 Fetching {symbol} ...")

            df = capital_market.price_volume_and_deliverable_position_data(
                symbol=symbol,
                from_date=start.strftime("%d-%m-%Y"),
                to_date=end.strftime("%d-%m-%Y"),
            )

            if df.empty:
                print(f"⚠️ No price data for {symbol}")
                failed.append(symbol)
                continue

            df.columns = df.columns.str.lower().str.strip()

            # 🔥 Keep only EQ series - Equity shares since they are available for continuous trading during the market hours
            if "series" in df.columns:
                df = df[df["series"].str.upper() == "EQ"].copy()

                if df.empty:
                    print(f"⚠️ {symbol} has no EQ data")
                    failed.append(symbol)
                    continue

            df.to_csv(RAW_DIR / f"{symbol}.csv", index=False)
            print(f"✅ Saved price data: {symbol}")

            # --- Fetch News ---
            news_df = fetch_news_sentiment(symbol)

            if not news_df.empty:
                news_df.to_csv(NEWS_DIR / f"{symbol}_news.csv", index=False)
                print(f"📰 News saved: {symbol}")

        except Exception as e:
            print(f"❌ Failed: {symbol} | {e}")
            failed.append(symbol)

        time.sleep(1)

    print("\n✅ Data Fetch Completed.")
    print(f"✅ Success: {len(symbols) - len(failed)} | ❌ Failed: {len(failed)}")

    if failed:
        pd.DataFrame(failed, columns=["Symbol"]).to_csv(
            DATA_DIR / "failed_tickers.csv", index=False
        )
        print("⚠️ Failed tickers saved.")

<IPython.core.display.Javascript object>

In [9]:
# ==========================================================
# 🔹 STEP 4: Fetch Concall Highlights
# ==========================================================

HEADERS = {"User-Agent": "Mozilla/5.0"}


def scrape_concall(ticker):

    url = f"https://www.screener.in/company/{ticker}/concall/"
    r = requests.get(url, headers=HEADERS)

    if r.status_code != 200:
        return None

    soup = BeautifulSoup(r.text, "html.parser")

    # Updated selector
    blocks = soup.select(".concall-highlight, .highlight")

    if len(blocks) == 0:
        return None

    records = []

    for b in blocks:
        try:
            date_text = b.find("span").text.strip()
            date_val = pd.to_datetime(date_text, errors="coerce")

            text = b.get_text(separator=" ").strip()

            records.append({"ticker": ticker, "date": date_val, "concall_text": text})

        except:
            continue

    return pd.DataFrame(records)

<IPython.core.display.Javascript object>

In [10]:
# --- STEP 5: Execute ---
START_DATE = date(2020, 1, 1)
END_DATE = date.today()

fetch_all_nselib_data(tickers, START_DATE, END_DATE)

print("\n🎯 Price + News data ready for Phase 2.")

📦 Fetching ABB ...
✅ Saved price data: ABB
📰 News saved: ABB
📦 Fetching ADANIENSOL ...
✅ Saved price data: ADANIENSOL
📰 News saved: ADANIENSOL
📦 Fetching ADANIENT ...
✅ Saved price data: ADANIENT
📰 News saved: ADANIENT
📦 Fetching ADANIGREEN ...
✅ Saved price data: ADANIGREEN
📰 News saved: ADANIGREEN
📦 Fetching ADANIPORTS ...
✅ Saved price data: ADANIPORTS
📰 News saved: ADANIPORTS
📦 Fetching ADANIPOWER ...
✅ Saved price data: ADANIPOWER
📰 News saved: ADANIPOWER
📦 Fetching AMBUJACEM ...
✅ Saved price data: AMBUJACEM
📰 News saved: AMBUJACEM
📦 Fetching APOLLOHOSP ...
✅ Saved price data: APOLLOHOSP
📰 News saved: APOLLOHOSP
📦 Fetching ASIANPAINT ...
✅ Saved price data: ASIANPAINT
📰 News saved: ASIANPAINT
📦 Fetching DMART ...
✅ Saved price data: DMART
📰 News saved: DMART
📦 Fetching AXISBANK ...
✅ Saved price data: AXISBANK
📰 News saved: AXISBANK
📦 Fetching BAJAJ-AUTO ...
✅ Saved price data: BAJAJ-AUTO
📰 News saved: BAJAJ-AUTO
📦 Fetching BAJFINANCE ...
✅ Saved price data: BAJFINANCE
📰 News sav

<IPython.core.display.Javascript object>

In [11]:
# --- STEP 6: Execute Concall Scraping---
print("🗣 Fetching concall highlights...")

for ticker in tickers:

    df_concall = scrape_concall(ticker)

    if df_concall is not None and not df_concall.empty:
        df_concall.to_csv(CONCALL_DIR / f"{ticker}_concall.csv", index=False)

    time.sleep(1)

print("✅ Concall scraping completed.")

🗣 Fetching concall highlights...
✅ Concall scraping completed.


<IPython.core.display.Javascript object>

In [12]:
print("Total RAW files:", len(list(RAW_DIR.glob("*.csv"))))

Total RAW files: 100


<IPython.core.display.Javascript object>